
# Roxy notebook example: Sequence-inferred structural propensity descriptors

This notebook is a **reference implementation example** for the **sequence-inferred structural propensity descriptor family** in Roxy.

These descriptors do **not** use experimental or predicted 3D structures. Instead, they summarize **structural tendencies inferred directly from sequence** using residue-level propensities and local sequence patterns.

## Covered outputs

This notebook implements examples such as:

- mean alpha-helix propensity
- mean beta-sheet propensity
- mean turn propensity
- fraction of helix-favoring residues
- fraction of sheet-favoring residues
- fraction of turn-favoring residues
- local windowed helix / sheet / turn propensities
- maximum local helix / sheet / turn tendency
- structural propensity amplitudes
- helix-sheet balance
- turn-vs-secondary-structure balance
- counts of windows above propensity thresholds
- class-style implementation for later migration into Roxy

The notebook is designed as a **clean teaching implementation** so it can later become part of the real Roxy package.


In [1]:

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "ssp_1",
            "ssp_2",
            "ssp_3",
            "ssp_4",
            "ssp_5",
            "ssp_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,ssp_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,ssp_2,GGGGGGGGGGGGGGG,B
2,ssp_3,KRRKRRKRRKRRDDDDEE,A
3,ssp_4,ACDEFGHIKLMNPQRSTVWY,B
4,ssp_5,PPPPGSSSSSTTTTNNQQQ,A
5,ssp_6,MSTNPKPQRITLKDGNKVELV,B


## Constants and residue-level structural propensity scales

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Chou-Fasman-like propensities
HELIX_PROP = {
    "A": 1.45, "C": 0.77, "D": 1.01, "E": 1.53, "F": 1.12,
    "G": 0.53, "H": 1.24, "I": 1.00, "K": 1.07, "L": 1.34,
    "M": 1.20, "N": 0.73, "P": 0.59, "Q": 1.17, "R": 0.79,
    "S": 0.79, "T": 0.82, "V": 1.14, "W": 1.14, "Y": 0.61,
}

SHEET_PROP = {
    "A": 0.97, "C": 1.30, "D": 0.54, "E": 0.37, "F": 1.28,
    "G": 0.81, "H": 0.71, "I": 1.60, "K": 0.74, "L": 1.22,
    "M": 1.67, "N": 0.65, "P": 0.62, "Q": 1.23, "R": 0.90,
    "S": 0.72, "T": 1.20, "V": 1.65, "W": 1.19, "Y": 1.29,
}

TURN_PROP = {
    "A": 0.66, "C": 1.19, "D": 1.46, "E": 0.74, "F": 0.60,
    "G": 1.56, "H": 0.95, "I": 0.47, "K": 1.01, "L": 0.59,
    "M": 0.60, "N": 1.56, "P": 1.52, "Q": 0.98, "R": 0.95,
    "S": 1.43, "T": 0.96, "V": 0.50, "W": 0.96, "Y": 1.14,
}

HELIX_FAVORING = {aa for aa, val in HELIX_PROP.items() if val >= 1.0}
SHEET_FAVORING = {aa for aa, val in SHEET_PROP.items() if val >= 1.0}
TURN_FAVORING = {aa for aa, val in TURN_PROP.items() if val >= 1.0}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def windows(seq: str, size: int):
    if len(seq) < size:
        return []
    return [seq[i:i+size] for i in range(len(seq) - size + 1)]


def scale_mean(seq: str, scale: dict) -> float:
    if len(seq) == 0:
        return np.nan
    return float(np.mean([scale[aa] for aa in seq]))


def fraction_from_group(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def profile_stats(values):
    if len(values) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "amplitude": np.nan,
        }
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=0)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "amplitude": float(np.max(values) - np.min(values)),
    }


def fraction_above_threshold(values, threshold: float) -> float:
    if len(values) == 0:
        return np.nan
    return float(np.mean(np.array(values) > threshold))


## Core descriptor function

In [5]:

def structural_propensity_descriptors(seq: str, window_sizes=(5, 7), threshold=1.0) -> dict:
    seq = clean_sequence(seq)

    out = {
        "ssp_length": len(seq),
        "ssp_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    # Global means
    helix_vals = [HELIX_PROP[aa] for aa in seq]
    sheet_vals = [SHEET_PROP[aa] for aa in seq]
    turn_vals = [TURN_PROP[aa] for aa in seq]

    out["ssp_helix_mean"] = float(np.mean(helix_vals))
    out["ssp_sheet_mean"] = float(np.mean(sheet_vals))
    out["ssp_turn_mean"] = float(np.mean(turn_vals))

    out["ssp_helix_std"] = float(np.std(helix_vals, ddof=0))
    out["ssp_sheet_std"] = float(np.std(sheet_vals, ddof=0))
    out["ssp_turn_std"] = float(np.std(turn_vals, ddof=0))

    # Global favoring fractions
    out["ssp_helix_favoring_fraction"] = fraction_from_group(seq, HELIX_FAVORING)
    out["ssp_sheet_favoring_fraction"] = fraction_from_group(seq, SHEET_FAVORING)
    out["ssp_turn_favoring_fraction"] = fraction_from_group(seq, TURN_FAVORING)

    # Global balances
    out["ssp_helix_sheet_balance"] = out["ssp_helix_mean"] - out["ssp_sheet_mean"]
    out["ssp_turn_vs_secondary_balance"] = out["ssp_turn_mean"] - ((out["ssp_helix_mean"] + out["ssp_sheet_mean"]) / 2.0)

    # Sliding-window structural propensity profiles
    for window_size in window_sizes:
        ws = windows(seq, window_size)

        helix_profile = [scale_mean(w, HELIX_PROP) for w in ws]
        sheet_profile = [scale_mean(w, SHEET_PROP) for w in ws]
        turn_profile = [scale_mean(w, TURN_PROP) for w in ws]

        profile_map = {
            "helix": helix_profile,
            "sheet": sheet_profile,
            "turn": turn_profile,
        }

        for name, values in profile_map.items():
            stats = profile_stats(values)
            prefix = f"ssp_w{window_size}_{name}"

            out[f"{prefix}_mean"] = stats["mean"]
            out[f"{prefix}_std"] = stats["std"]
            out[f"{prefix}_min"] = stats["min"]
            out[f"{prefix}_max"] = stats["max"]
            out[f"{prefix}_amplitude"] = stats["amplitude"]
            out[f"{prefix}_high_fraction"] = fraction_above_threshold(values, threshold=threshold)

        # Window-level balances
        if len(helix_profile) > 0 and len(sheet_profile) > 0:
            helix_sheet_profile = np.array(helix_profile) - np.array(sheet_profile)
            out[f"ssp_w{window_size}_helix_sheet_balance_mean"] = float(np.mean(helix_sheet_profile))
            out[f"ssp_w{window_size}_helix_sheet_balance_max"] = float(np.max(helix_sheet_profile))
            out[f"ssp_w{window_size}_helix_sheet_balance_min"] = float(np.min(helix_sheet_profile))
        else:
            out[f"ssp_w{window_size}_helix_sheet_balance_mean"] = np.nan
            out[f"ssp_w{window_size}_helix_sheet_balance_max"] = np.nan
            out[f"ssp_w{window_size}_helix_sheet_balance_min"] = np.nan

    return out


## Functional usage on one sequence

In [6]:

example = structural_propensity_descriptors(df_demo.loc[0, "sequence"], window_sizes=(5, 7), threshold=1.0)
list(example.items())[:20]


[('ssp_length', 24),
 ('ssp_valid_residue_count', 24),
 ('ssp_helix_mean', 1.0054166666666668),
 ('ssp_sheet_mean', 1.1304166666666668),
 ('ssp_turn_mean', 0.8791666666666665),
 ('ssp_helix_std', 0.2435839890514609),
 ('ssp_sheet_std', 0.310436240993577),
 ('ssp_turn_std', 0.3496416816621776),
 ('ssp_helix_favoring_fraction', 0.5833333333333334),
 ('ssp_sheet_favoring_fraction', 0.5833333333333334),
 ('ssp_turn_favoring_fraction', 0.2916666666666667),
 ('ssp_helix_sheet_balance', -0.125),
 ('ssp_turn_vs_secondary_balance', -0.1887500000000003),
 ('ssp_w5_helix_mean', 1.0030999999999997),
 ('ssp_w5_helix_std', 0.12721238147287398),
 ('ssp_w5_helix_min', 0.7719999999999999),
 ('ssp_w5_helix_max', 1.2520000000000002),
 ('ssp_w5_helix_amplitude', 0.4800000000000003),
 ('ssp_w5_helix_high_fraction', 0.55),
 ('ssp_w5_sheet_mean', 1.1220999999999999)]

## Apply structural propensity descriptors to the full dataset

In [7]:

df_ssp = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(
            lambda x: structural_propensity_descriptors(x, window_sizes=(5, 7), threshold=1.0)
        ).apply(pd.Series),
    ],
    axis=1,
)

df_ssp.head()


,sequence_id,sequence,label,ssp_length,ssp_valid_residue_count,ssp_helix_mean,ssp_sheet_mean,ssp_turn_mean,ssp_helix_std,ssp_sheet_std,...,ssp_w7_sheet_high_fraction,ssp_w7_turn_mean,ssp_w7_turn_std,ssp_w7_turn_min,ssp_w7_turn_max,ssp_w7_turn_amplitude,ssp_w7_turn_high_fraction,ssp_w7_helix_sheet_balance_mean,ssp_w7_helix_sheet_balance_max,ssp_w7_helix_sheet_balance_min
0,ssp_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,1.005417,1.130417,0.879167,2.435840e-01,3.104362e-01,...,0.833333,0.895000,0.153106,0.695714,1.228571,0.532857,0.277778,-0.104206,0.077143,-0.262857
1,ssp_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.530000,0.810000,1.560000,1.110223e-16,2.220446e-16,...,0.000000,1.560000,0.000000,1.560000,1.560000,0.000000,1.000000,-0.280000,-0.280000,-0.280000
2,ssp_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.983333,0.725556,1.053333,2.272541e-01,1.878008e-01,...,0.000000,1.065357,0.107321,0.967143,1.250000,0.282857,0.500000,0.159167,0.584286,0.015714
3,ssp_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,1.002000,1.033000,0.991500,2.803141e-01,3.736322e-01,...,0.571429,0.997449,0.087888,0.825714,1.142857,0.317143,0.428571,-0.018878,0.238571,-0.245714
4,ssp_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.794211,0.877895,1.299474,1.879618e-01,2.601736e-01,...,0.230769,1.290769,0.142929,1.134286,1.500000,0.365714,1.000000,-0.101319,0.005714,-0.202857


## Inspect structural propensity descriptor columns

In [8]:

ssp_cols = [c for c in df_ssp.columns if c.startswith("ssp_") and c not in {"ssp_length", "ssp_valid_residue_count"}]
len(ssp_cols), ssp_cols[:18]


(53,
 ['ssp_helix_mean',
  'ssp_sheet_mean',
  'ssp_turn_mean',
  'ssp_helix_std',
  'ssp_sheet_std',
  'ssp_turn_std',
  'ssp_helix_favoring_fraction',
  'ssp_sheet_favoring_fraction',
  'ssp_turn_favoring_fraction',
  'ssp_helix_sheet_balance',
  'ssp_turn_vs_secondary_balance',
  'ssp_w5_helix_mean',
  'ssp_w5_helix_std',
  'ssp_w5_helix_min',
  'ssp_w5_helix_max',
  'ssp_w5_helix_amplitude',
  'ssp_w5_helix_high_fraction',
  'ssp_w5_sheet_mean'])

In [9]:

df_ssp[
    [
        "sequence_id",
        "ssp_helix_mean",
        "ssp_sheet_mean",
        "ssp_turn_mean",
        "ssp_helix_sheet_balance",
        "ssp_w5_helix_max",
        "ssp_w5_sheet_max",
        "ssp_w7_turn_high_fraction",
    ]
]


,sequence_id,ssp_helix_mean,ssp_sheet_mean,ssp_turn_mean,ssp_helix_sheet_balance,ssp_w5_helix_max,ssp_w5_sheet_max,ssp_w7_turn_high_fraction
0,ssp_1,1.005417,1.130417,0.879167,-0.125000,1.252,1.384,0.277778
1,ssp_2,0.530000,0.810000,1.560000,-0.280000,0.530,0.810,1.000000
2,ssp_3,0.983333,0.725556,1.053333,0.257778,1.218,0.868,0.500000
3,ssp_4,1.002000,1.033000,0.991500,-0.031000,1.176,1.210,0.428571
4,ssp_5,0.794211,0.877895,1.299474,-0.083684,0.994,1.104,1.000000
5,ssp_6,0.974762,0.987619,1.022857,-0.012857,1.244,1.230,0.666667


## Dataset-level summary

In [10]:

ssp_summary = (
    df_ssp[ssp_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

ssp_summary.head(15)


,descriptor,mean_value
0,ssp_w5_turn_max,1.378667
1,ssp_w7_turn_max,1.327381
2,ssp_w5_turn_mean,1.151730
3,ssp_w7_turn_mean,1.147540
4,ssp_turn_mean,1.134388
5,ssp_w5_sheet_max,1.101000
6,ssp_w5_helix_max,1.069000
7,ssp_w7_sheet_max,1.049048
8,ssp_w7_turn_min,1.016667
9,ssp_w7_helix_max,0.989762


## Sanity checks

In [11]:

assert "ssp_helix_mean" in df_ssp.columns
assert "ssp_sheet_mean" in df_ssp.columns
assert "ssp_turn_mean" in df_ssp.columns
assert "ssp_w5_helix_amplitude" in df_ssp.columns
assert "ssp_w7_sheet_high_fraction" in df_ssp.columns
assert "ssp_w5_helix_sheet_balance_mean" in df_ssp.columns
assert df_ssp["ssp_length"].min() > 0

print(f"Number of structural propensity descriptor columns: {len(ssp_cols)}")
print("Structural propensity descriptor checks passed.")


Number of structural propensity descriptor columns: 53
Structural propensity descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class StructuralPropensityDescriptors:
    """Example class-style structural propensity implementation for later migration into Roxy."""

    def __init__(self, window_sizes=(5, 7), threshold=1.0):
        self.window_sizes = tuple(window_sizes)
        self.threshold = float(threshold)

    def transform_sequence(self, seq: str) -> dict:
        return structural_propensity_descriptors(
            seq,
            window_sizes=self.window_sizes,
            threshold=self.threshold,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


ssp_transformer = StructuralPropensityDescriptors(window_sizes=(5, 7), threshold=1.0)
ssp_matrix = ssp_transformer.transform(df_demo["sequence"].tolist())
ssp_matrix.head()


,ssp_length,ssp_valid_residue_count,ssp_helix_mean,ssp_sheet_mean,ssp_turn_mean,ssp_helix_std,ssp_sheet_std,ssp_turn_std,ssp_helix_favoring_fraction,ssp_sheet_favoring_fraction,...,ssp_w7_sheet_high_fraction,ssp_w7_turn_mean,ssp_w7_turn_std,ssp_w7_turn_min,ssp_w7_turn_max,ssp_w7_turn_amplitude,ssp_w7_turn_high_fraction,ssp_w7_helix_sheet_balance_mean,ssp_w7_helix_sheet_balance_max,ssp_w7_helix_sheet_balance_min
0,24,24,1.005417,1.130417,0.879167,2.435840e-01,3.104362e-01,3.496417e-01,0.583333,0.583333,...,0.833333,0.895000,0.153106,0.695714,1.228571,0.532857,0.277778,-0.104206,0.077143,-0.262857
1,15,15,0.530000,0.810000,1.560000,1.110223e-16,2.220446e-16,4.440892e-16,0.000000,0.000000,...,0.000000,1.560000,0.000000,1.560000,1.560000,0.000000,1.000000,-0.280000,-0.280000,-0.280000
2,18,18,0.983333,0.725556,1.053333,2.272541e-01,1.878008e-01,2.298309e-01,0.555556,0.000000,...,0.000000,1.065357,0.107321,0.967143,1.250000,0.282857,0.500000,0.159167,0.584286,0.015714
3,20,20,1.002000,1.033000,0.991500,2.803141e-01,3.736322e-01,3.577188e-01,0.600000,0.500000,...,0.571429,0.997449,0.087888,0.825714,1.142857,0.317143,0.428571,-0.018878,0.238571,-0.245714
4,19,19,0.794211,0.877895,1.299474,1.879618e-01,2.601736e-01,2.565453e-01,0.157895,0.368421,...,0.230769,1.290769,0.142929,1.134286,1.500000,0.365714,1.000000,-0.101319,0.005714,-0.202857


## Merge transformer output back to the dataset

In [13]:

df_ssp_class = pd.concat([df_demo, ssp_matrix], axis=1)
df_ssp_class.head()


,sequence_id,sequence,label,ssp_length,ssp_valid_residue_count,ssp_helix_mean,ssp_sheet_mean,ssp_turn_mean,ssp_helix_std,ssp_sheet_std,...,ssp_w7_sheet_high_fraction,ssp_w7_turn_mean,ssp_w7_turn_std,ssp_w7_turn_min,ssp_w7_turn_max,ssp_w7_turn_amplitude,ssp_w7_turn_high_fraction,ssp_w7_helix_sheet_balance_mean,ssp_w7_helix_sheet_balance_max,ssp_w7_helix_sheet_balance_min
0,ssp_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,1.005417,1.130417,0.879167,2.435840e-01,3.104362e-01,...,0.833333,0.895000,0.153106,0.695714,1.228571,0.532857,0.277778,-0.104206,0.077143,-0.262857
1,ssp_2,GGGGGGGGGGGGGGG,B,15,15,0.530000,0.810000,1.560000,1.110223e-16,2.220446e-16,...,0.000000,1.560000,0.000000,1.560000,1.560000,0.000000,1.000000,-0.280000,-0.280000,-0.280000
2,ssp_3,KRRKRRKRRKRRDDDDEE,A,18,18,0.983333,0.725556,1.053333,2.272541e-01,1.878008e-01,...,0.000000,1.065357,0.107321,0.967143,1.250000,0.282857,0.500000,0.159167,0.584286,0.015714
3,ssp_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,1.002000,1.033000,0.991500,2.803141e-01,3.736322e-01,...,0.571429,0.997449,0.087888,0.825714,1.142857,0.317143,0.428571,-0.018878,0.238571,-0.245714
4,ssp_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.794211,0.877895,1.299474,1.879618e-01,2.601736e-01,...,0.230769,1.290769,0.142929,1.134286,1.500000,0.365714,1.000000,-0.101319,0.005714,-0.202857



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move propensity scales into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/structure_propensity.py`
- expose a class such as `StructuralPropensityDescriptors`
- allow configurable:
  - window sizes
  - selected propensity families
  - threshold used for "high" propensity windows
- add tests for:
  - empty sequences
  - proline/glycine-rich turn-prone sequences
  - hydrophobic helix-prone sequences
  - mixed sequences with balanced helix and sheet tendencies
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_ssp.to_csv("demo_structural_propensity_descriptors.csv", index=False)
